# 587Ah 电芯：不同循环深度（DOD）的容量保持率对比

**工况：0.5P 恒功率 · 298.15 K · t_factor 老化加速 · 定电压窗口控制**

对比两种循环深度在 **相同累计 Ah 通量** 下的容量保持率差异：

| 方案 | SOC 窗口 | DOD | 单圈放电量 | 容量对标方式 |
|---|---|---|---|---|
| A 浅充放 | 90–100% | 10% | ≈ 0.1·587 ≈ 58.7 Ah | 每隔 `REF_INTERVAL` 圈插入 1 次 100%SOC 满充放参考循环 |
| B 满充放 | 0–100% | 100% | ≈ 587 Ah | 每圈满放即为真实剩余容量 |

> **为什么浅充放需要参考循环**：90–100% 的浅循环永远不满放，其自身放电量≈58.7 Ah
> 反映的是窗口宽度而非剩余容量，无法读出衰减；故每隔若干圈插入一次满放循环，用满放容量对标。
>
> **相同通量对比**：浅充放 DOD 只有满充放的 1/10，要累积到相同 Ah 通量需 ~10× 圈数。
> 本 notebook 用项目既有的 `t_factor` 老化加速（每仿真圈 ≈ `t_factor` 实测圈的衰减速率），
> 横轴统一用 **等效累计 Ah 通量**，两方案同 `t_factor`，等通量处比较公平。


In [ ]:
# --- BatteryProject 路径自动定位 (无论从 root 还是 examples/ 启动 notebook 都能工作) ---
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
for _cand in (_HERE, *_HERE.parents):
    if (_cand / "src" / "easy_imports.py").exists() and (_cand / "pyproject.toml").exists():
        PROJECT_ROOT = _cand
        break
else:
    raise RuntimeError(f"Cannot locate BatteryProject root from {_HERE}")

WORKSPACE_ROOT = PROJECT_ROOT.parent
PARAMS_DIR = WORKSPACE_ROOT / "params"
for _p in (PROJECT_ROOT, WORKSPACE_ROOT, PARAMS_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

OUT_DIR = PROJECT_ROOT / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PARAMS_DIR:  ", PARAMS_DIR)
print("OUT_DIR:     ", OUT_DIR)

In [ ]:
from src.easy_imports import (
    apply_pybamm_runtime_limits,
    build_power_step,
    compute_cycle_energies,
    run_aging_with_dryout,
)
from src.plotting import plot_analysis

apply_pybamm_runtime_limits()

import matplotlib.pyplot as plt
import numpy as np
import pybamm

from params587 import get_hithium_params  # 587Ah 海辰参数（params/ 已在 sys.path）

pybamm.set_logging_level("WARNING")
print("依赖导入成功")


## 第 1 步：老化模型 + 参数构造

每个方案/标定都**新建一份 params 和 model**，避免 OKane2022 被就地改写污染。


In [ ]:
def build_aging_model() -> pybamm.BaseModel:
    options = {
        "SEI": "ec reaction limited",
        "SEI porosity change": "true",
        "lithium plating": "irreversible",
        "lithium plating porosity change": "true",
        "particle mechanics": ("swelling and cracking", "swelling only"),
        "SEI on cracks": "true",
        "loss of active material": "stress-driven",
        "contact resistance": "true",
    }
    return pybamm.lithium_ion.DFN(options)

def build_params(t_factor, temperature_k):
    # OKane2022 基底 + 587Ah 海辰参数；每次新建，杜绝跨方案污染
    params = pybamm.ParameterValues("OKane2022")
    params.update(get_hithium_params(t_factor=t_factor, temperature=temperature_k),
                  check_already_exists=False)
    return params

_p = build_params(1, 298.15)
NOMINAL = float(_p["Nominal cell capacity [A.h]"])
print(f"标称容量: {NOMINAL:.0f} Ah")


## 第 2 步：仿真配置

`QUICK_DEMO=True`：演示规模（分钟级，`REF_INTERVAL=50`，对标点可见）；
`QUICK_DEMO=False`：按工况书写的 `REF_INTERVAL=500`、更长圈数（运行更久）。
两种规模的**通量等效关系与对标机制完全相同**。


In [ ]:
# ---- 工况 ----
TEMPERATURE_K = 298.15
T_FACTOR = 50                       # 老化加速因子（= 实测圈/仿真圈衰减速率）
REF_VOLTAGE = 3.2                   # 标称电压，用于功率<->C 折算
POWER_W = 0.5 * NOMINAL * REF_VOLTAGE   # 0.5P ≈ 0.5C 等效恒功率 ≈ 939 W
UPPER_V = 3.65                      # 100% SOC 截止电压
LOWER_V = 2.5                       # 0% SOC 截止电压
PARTIAL_DOD = 0.10                  # 浅充放窗口深度 = 10% (90-100% SOC)

# ---- 规模 / 对标间隔 ----
QUICK_DEMO = True
if QUICK_DEMO:
    REF_INTERVAL = 50              # 每 50 个浅循环插一次满放参考（演示）
    FULL_CYCLES = 20              # 满充放仿真圈数
else:
    REF_INTERVAL = 500            # 工况书写值
    FULL_CYCLES = 100

# 浅充放 DOD=10%，等通量需 10× 圈数
SHALLOW_CYCLES = FULL_CYCLES * 10
N_REF_BLOCKS = SHALLOW_CYCLES // REF_INTERVAL   # 浅充放分块数 = 满放对标点数

# ---- 求解器 / 网格 ----
solver = pybamm.IDAKLUSolver(rtol=1e-6, atol=1e-6)
var_pts = {"x_n": 5, "x_s": 5, "x_p": 5, "r_n": 10, "r_p": 10}

print(f"恒功率 POWER_W = {POWER_W:.1f} W (0.5P)")
print(f"满充放: {FULL_CYCLES} 仿真圈   浅充放: {SHALLOW_CYCLES} 仿真圈 "
      f"(分 {N_REF_BLOCKS} 块, 每块 {REF_INTERVAL} 浅圈 + 1 满放对标)")
print(f"等效满放圈数(通量/标称) ≈ {FULL_CYCLES * T_FACTOR} 实测圈")


## 第 3 步：标定 90% SOC 窗口（电压 → 容量控制）

> **为什么不用纯电压截止控制 90–100% 窗口**：LFP 电压平台极平坦，0.5P 放电起始
> 有一个 IR+表面浓差的瞬时电压下陷（落到 ~3.29 V 后回升）。若用 `Discharge until V_90`，
> 事件会在这个起始下陷处立即触发，窗口塌缩到 ~1% DOD（实测仅放出 ~5 Ah ≠ 10%）。
> 因此本 notebook 用 **定容量（恒功率定时）放电** 实现 10% DOD 窗口，顶端由每圈
> 满充+CV 重锚到 100%——既稳健、又真正落在 90–100%。下面同时**标定并报告该窗口下端
> 电压 `V_90`** 作为诊断（即"电压窗口"信息），但控制量是容量。


In [ ]:
# 放出 10% 容量所需的恒功率时长（恒功率近似）
T_PARTIAL_H = PARTIAL_DOD * NOMINAL * REF_VOLTAGE / POWER_W

def calibrate_v90():
    model = pybamm.lithium_ion.DFN()          # 标定用基础 DFN，快
    params = build_params(1, TEMPERATURE_K)
    exp = pybamm.Experiment(
        [f"Charge at {POWER_W:.1f}W until {UPPER_V} V",
         f"Hold at {UPPER_V} V until C/20",
         f"Discharge at {POWER_W:.1f}W for {T_PARTIAL_H:.3f} hours"],
        temperature=TEMPERATURE_K,
    )
    sol = pybamm.Simulation(model, experiment=exp, parameter_values=params,
                            solver=solver, var_pts=var_pts).solve(showprogress=False)
    return float(sol["Terminal voltage [V]"].entries[-1])

V_90 = calibrate_v90()
print(f"浅循环放电时长 T_PARTIAL_H = {T_PARTIAL_H:.3f} h  (≈ {PARTIAL_DOD*100:.0f}% DOD)")
print(f"诊断：该窗口下端电压 V_90 ≈ {V_90:.4f} V")


## 第 4 步：构造两种实验

模型初始为放电态，故每个循环**先充后放**（与初始态无关，稳健）。

- 满充放参考循环：满充(→3.65V)+CV 保持至 C/20，再满放(→2.5V)。满放电量=真实剩余容量。
- 浅充放循环：满充(→3.65V)+CV 重锚 100%，再定容量放出 10%（`T_PARTIAL_H` 恒功率定时，
  带 2.5V 安全下限）。窗口稳定落在 90–100%。
- 浅充放每块 = 1 次满放对标 + `REF_INTERVAL` 个浅循环（块首对标，首块即 BOL 基准）。


In [ ]:
def reference_full_cycle():
    return (
        build_power_step("Charge", POWER_W, UPPER_V, NOMINAL),
        f"Hold at {UPPER_V} V until C/20",
        build_power_step("Discharge", POWER_W, LOWER_V, NOMINAL),
    )

def partial_cycle():
    return (
        build_power_step("Charge", POWER_W, UPPER_V, NOMINAL),
        f"Hold at {UPPER_V} V until C/20",
        f"Discharge at {POWER_W:.1f}W for {T_PARTIAL_H:.3f} hours or until {LOWER_V} V",
    )

def make_experiment_full(n):
    return pybamm.Experiment([reference_full_cycle()] * n, temperature=TEMPERATURE_K)

def make_experiment_shallow(cpb):
    # 块首插 1 次满放对标，随后 cpb 个浅循环
    steps = [reference_full_cycle()] + [partial_cycle()] * cpb
    return pybamm.Experiment(steps, temperature=TEMPERATURE_K)

print("满放参考循环步:", reference_full_cycle())
print("浅充放循环步: ", partial_cycle())


## 第 5 步：运行两种方案

`run_aging_with_dryout`（tracker=None）= 纯分块老化；块间通过 `starting_solution`
链式累计，`sol_list[-1]` 含全部循环。t_factor 经 `build_params` 注入衰减速率。


In [ ]:
def run_full():
    model = build_aging_model()
    params = build_params(T_FACTOR, TEMPERATURE_K)
    return run_aging_with_dryout(
        model, params, make_experiment_full, solver, var_pts,
        n_blocks=1, cycles_per_block=FULL_CYCLES,
        t_factor=T_FACTOR, temperature=TEMPERATURE_K, showprogress=False,
    )

def run_shallow():
    model = build_aging_model()
    params = build_params(T_FACTOR, TEMPERATURE_K)
    return run_aging_with_dryout(
        model, params, make_experiment_shallow, solver, var_pts,
        n_blocks=N_REF_BLOCKS, cycles_per_block=REF_INTERVAL,
        t_factor=T_FACTOR, temperature=TEMPERATURE_K, showprogress=False,
    )

print(">>> 运行【满充放 0-100%】...")
sol_full = run_full()
print(">>> 运行【浅充放 90-100%】...")
sol_shallow = run_shallow()
print(f"完成：满放 {len(sol_full[-1].cycles)} 圈，浅放 {len(sol_shallow[-1].cycles)} 圈(含对标)")


## 第 6 步：按等效累计 Ah 通量对比容量保持率

- 满充放：每圈放电容量即剩余容量，逐圈给出保持率。
- 浅充放：用满放参考循环（放电容量 > 0.5·标称）测真实剩余容量；其余浅圈只计入通量。
- 横轴 = 等效累计 Ah 通量 = `cumsum(每圈放电容量) × t_factor`，两方案同口径。


In [ ]:
def cumulative_throughput_kAh(cap):
    return np.cumsum(cap) * T_FACTOR / 1000.0   # kAh，等效实测通量

# --- 满充放：逐圈即容量 ---
rf = compute_cycle_energies(sol_full[-1])
cap_f = rf["discharge_cap"]
thr_f = cumulative_throughput_kAh(cap_f)
ret_f = cap_f / cap_f[0] * 100.0

# --- 浅充放：仅取满放参考圈测容量；通量含全部圈 ---
rs = compute_cycle_energies(sol_shallow[-1])
cap_s_all = rs["discharge_cap"]
thr_s_all = cumulative_throughput_kAh(cap_s_all)
ref_mask = cap_s_all > 0.5 * NOMINAL          # 满放对标圈判别
cap_ref = cap_s_all[ref_mask]
thr_ref = thr_s_all[ref_mask]
ret_ref = cap_ref / cap_ref[0] * 100.0

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.8))
axL.plot(thr_f, ret_f, "s-", color="tab:gray", lw=2, ms=4, label="满充放 0-100%")
axL.plot(thr_ref, ret_ref, "o-", color="tab:red", lw=2, ms=6, label="浅充放 90-100% (满放对标)")
axL.set_xlabel("等效累计放电通量 [kAh]")
axL.set_ylabel("容量保持率 [%]")
axL.set_title("相同通量下的容量保持率")
axL.legend(); axL.grid(alpha=0.3, ls="--")

axR.plot(thr_f, cap_f, "s-", color="tab:gray", lw=2, ms=4, label="满充放 0-100%")
axR.plot(thr_ref, cap_ref, "o-", color="tab:blue", lw=2, ms=6, label="浅充放 90-100% (满放对标)")
axR.set_xlabel("等效累计放电通量 [kAh]")
axR.set_ylabel("放电容量 [Ah]")
axR.set_title("剩余容量")
axR.legend(); axR.grid(alpha=0.3, ls="--")

fig.tight_layout()
fig.savefig(str(OUT_DIR / "depth_capacity_retention.png"), dpi=150, bbox_inches="tight")
plt.show()

# --- 等通量处差异（插值到公共通量网格）---
if ret_ref.size >= 2 and ret_f.size >= 2:
    x_max = min(thr_f[-1], thr_ref[-1])
    xg = np.linspace(max(thr_f[0], thr_ref[0]), x_max, 20)
    rf_i = np.interp(xg, thr_f, ret_f)
    rs_i = np.interp(xg, thr_ref, ret_ref)
    print(f"末点(≈{x_max:.1f} kAh) 保持率：满放 {rf_i[-1]:.2f}%  vs  浅放 {rs_i[-1]:.2f}%  "
          f"(Δ {rs_i[-1]-rf_i[-1]:+.2f} pp)")
    print(f"等通量区间内浅放相对满放平均高 {np.mean(rs_i - rf_i):+.2f} pp")
else:
    print("对标点不足以做等通量插值；增大 FULL_CYCLES 或减小 REF_INTERVAL。")


## 第 7 步：老化内部参数展示

`plot_analysis` 综合面板（2×4）逐方案展示老化机理内参：SOH 衰减、充放电曲线、
负极电解液最小浓度（干涸/传输受限）、析锂过电位、充电末端表面计量比、析锂项分解、
活性锂损失(LLI)、孔隙率分布。对比两方案可看出 90–100% 浅充放 vs 0–100% 满充放在
SEI/析锂/LAM/孔隙率上的差异来源。


In [ ]:
cases = [("满充放 0-100%", sol_full[-1]), ("浅充放 90-100%", sol_shallow[-1])]
for label, sol in cases:
    print(f"\n=== 老化内部参数：{label} ===")
    fig, _ = plot_analysis(sol, t_factor=T_FACTOR)
    fig.suptitle(label, y=1.02, fontsize=13)
    fname = "depth_internals_" + ("full" if "0-100" in label else "shallow") + ".png"
    fig.savefig(str(OUT_DIR / fname), dpi=150, bbox_inches="tight")
    plt.show()


## 备注

1. **预期趋势**：相同 Ah 通量下，90–100% 浅充放（小 DOD、不经历石墨低 SOC 相变与满放
   机械应力）通常衰减更慢、保持率更高；满充放衰减更快。
2. **窗口控制方式（与原始"定电压窗口"诉求的偏差）**：LFP 平台平坦 + 0.5P 起始电压下陷，
   使纯电压截止无法稳定切出 10% 窗口（会塌缩到 ~1% DOD）。故改用 **定容量恒功率定时放电**
   实现真正的 90–100% 窗口，并报告 `V_90` 作为该窗口下端电压诊断。若确需纯电压截止控制，
   可改回但需接受窗口失真。
3. **加速因子**：`T_FACTOR` 对两方案一致，仅压缩日历/循环里程；等通量比较的相对差异有效，
   绝对寿命数字需按真实 `t_factor=1` 标定后解读。
4. **放大规模**：设 `QUICK_DEMO=False` 启用 `REF_INTERVAL=500` 工况规模（运行更久）。
